# PhishGuard AI — Exploratory Data Analysis & Phishing URL Modeling

This notebook provides an in-depth exploratory data analysis and model training evaluation on the **UCI PhiUSIIL Phishing URL Dataset**.

### Contents:
1. **Data Ingestion**: Loading balanced legitimate and phishing URLs.
2. **Feature Engineering**: Computing 25 lexical, structural, and information-theoretic indicators.
3. **Exploratory Data Analysis**: Visualizing distributions, character ratios, and entropy.
4. **Model Benchmarking**: Comparing Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting.
5. **Feature Importance & Confusion Matrix**: Deep-dive into model explainability.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add root directory to sys.path
sys.path.append(os.path.abspath('..'))

from src.data_loader import DataLoader
from src.features import URLFeatureExtractor
from src.model import ModelManager, PhishGuardPredictor

sns.set_theme(style='whitegrid')
print('Environment initialized successfully.')

## 1. Load Processed Dataset
We load the cached balanced sample of URLs extracted from the UCI repository.

In [ ]:
data_loader = DataLoader(data_dir='../data')
df = data_loader.load_and_preprocess(sample_size=10000, random_state=42)
print(f'Total records: {len(df)}')
print(df['is_phishing'].value_counts())
df.head()

## 2. Feature Extraction
Extract 25 security-grounded features using `URLFeatureExtractor`.

In [ ]:
extractor = URLFeatureExtractor()
X = extractor.transform(df['url'])
y = df['is_phishing']
print('Extracted features shape:', X.shape)
X.head()

## 3. Exploratory Data Analysis

In [ ]:
eda_df = X.copy()
eda_df['Class'] = y.map({0: 'Legitimate', 1: 'Phishing'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=eda_df, x='Class', y='url_length', ax=axes[0], palette='Blues')
axes[0].set_title('URL Length Distribution')
axes[0].set_ylim(0, 160)

sns.kdeplot(data=eda_df, x='url_entropy', hue='Class', fill=True, ax=axes[1], palette='tab10')
axes[1].set_title('Shannon Entropy Distribution')
plt.tight_layout()
plt.show()

## 4. Multi-Model Benchmark & Evaluation

In [ ]:
X_train, X_test, y_train, y_test = data_loader.get_train_test_data(X, y, test_size=0.2, random_state=42)
mgr = ModelManager(models_dir='../models')
results = mgr.train_and_evaluate(X_train, X_test, y_train, y_test)

benchmarks = pd.DataFrame([
    {
        'Model': name,
        'Accuracy': d['accuracy'],
        'Precision': d['precision'],
        'Recall': d['recall'],
        'F1-Score': d['f1_score'],
        'ROC-AUC': d['roc_auc']
    }
    for name, d in results.items()
])
benchmarks

## 5. Feature Importance Analysis

In [ ]:
rf = results['Random Forest']['model']
fi = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
plt.figure(figsize=(10, 6))
fi.plot(kind='barh', color='#3b82f6')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.xlabel('Gini Importance')
plt.gca().invert_yaxis()
plt.show()